# 张量形状变化

In [1]:
import torch
from torch import nn

In [2]:
t = torch.arange(24)
print(t)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])


## 1.reshape 和 view

In [3]:
td1 = t.reshape(4,6)
print(td1)

tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17],
        [18, 19, 20, 21, 22, 23]])


In [5]:
td2 = t.reshape(2,3,4)
print(td2)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


In [7]:
# reshape 共享内存
td1[0][0] = 100
print(td2)

tensor([[[100,   1,   2,   3],
         [  4,   5,   6,   7],
         [  8,   9,  10,  11]],

        [[ 12,  13,  14,  15],
         [ 16,  17,  18,  19],
         [ 20,  21,  22,  23]]])


In [ ]:
td3 = t.reshape(6,-1)  # -1表示自行推断
print(td3)


tensor([[100,   1,   2,   3],
        [  4,   5,   6,   7],
        [  8,   9,  10,  11],
        [ 12,  13,  14,  15],
        [ 16,  17,  18,  19],
        [ 20,  21,  22,  23]])


view 要求内存连续， 修改会影响原tensor

In [13]:
td4 = t.view(3,8)
print(td4)
print(t)
td3[0][0] = 60
print(t)

tensor([[100,   1,   2,   3,   4,   5,   6,   7],
        [  8,   9,  10,  11,  12,  13,  14,  15],
        [ 16,  17,  18,  19,  20,  21,  22,  23]])
tensor([100,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
         14,  15,  16,  17,  18,  19,  20,  21,  22,  23])
tensor([60,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])


In [15]:
# 需要恢复后再view
t_contig = td2.contiguous().view(-1)
print(t_contig)

tensor([60,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])


核心一句话：

`view()` **只能用于连续张量（contiguous）**，不拷贝数据，只是修改元信息（shape/stride）；

`reshape()` **自动判断**：张量连续 → 等价 `view()`；张量不连续 → 底层拷贝一份新数据，返回新张量。

## 2.squeeze和unsqueeze

In [18]:
a = torch.zeros(1,3,1,5)
print(a)
print(a.shape)

tensor([[[[0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.]]]])
torch.Size([1, 3, 1, 5])


In [21]:
# 直接使用会删除所有维度为1的维度
b = a.squeeze()
print(b)
print(b.shape)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])
torch.Size([3, 5])


In [ ]:
# 传参可删除指定维度
c = a.squeeze(0)
print(c)
print(c.shape)



tensor([[[0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.]]])
torch.Size([3, 1, 5])


In [24]:
# squeeze() 默认只能删除维度大小等于 1 的轴，其它维度不会动
c = a.squeeze(1)
print(c)
print(c.shape)

tensor([[[[0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.]]]])
torch.Size([1, 3, 1, 5])


In [25]:
# 插入一个维度
d = c.unsqueeze(dim=0)
print(d.shape)
d = d.unsqueeze(dim=-1)
print(d.shape)

torch.Size([1, 1, 3, 1, 5])
torch.Size([1, 1, 3, 1, 5, 1])


## 3.premute 和 transpose

In [26]:
t = torch.randn(2,3,4)
print(t)
print(t.shape)

tensor([[[-0.7815,  1.3378, -0.2576, -0.6780],
         [ 0.4980, -1.5778,  0.5103,  0.3734],
         [ 0.1073,  0.5124, -0.7549,  0.6966]],

        [[ 0.8686,  0.1773, -0.3638, -1.4834],
         [ 0.5070,  0.1208, -1.4525,  1.2896],
         [ 0.2870, -1.1860, -0.5959,  0.5729]]])
torch.Size([2, 3, 4])


permute维度全部重排序

In [ ]:
a = t.permute(2,1,0)  # 把所有维度变化  2->0 1->1 0->2
print(a)

tensor([[[-0.7815,  0.8686],
         [ 0.4980,  0.5070],
         [ 0.1073,  0.2870]],

        [[ 1.3378,  0.1773],
         [-1.5778,  0.1208],
         [ 0.5124, -1.1860]],

        [[-0.2576, -0.3638],
         [ 0.5103, -1.4525],
         [-0.7549, -0.5959]],

        [[-0.6780, -1.4834],
         [ 0.3734,  1.2896],
         [ 0.6966,  0.5729]]])


In [28]:
# 换轴
b = t.transpose(1,2)
print(b)
print(b.shape)

tensor([[[-0.7815,  0.4980,  0.1073],
         [ 1.3378, -1.5778,  0.5124],
         [-0.2576,  0.5103, -0.7549],
         [-0.6780,  0.3734,  0.6966]],

        [[ 0.8686,  0.5070,  0.2870],
         [ 0.1773,  0.1208, -1.1860],
         [-0.3638, -1.4525, -0.5959],
         [-1.4834,  1.2896,  0.5729]]])
torch.Size([2, 4, 3])


In [29]:
# 二维矩阵转置
m = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(m.T)                        # (2,3) → (3,2)
print(m.t())                      # 同上，仅支持 2D

tensor([[1, 4],
        [2, 5],
        [3, 6]])
tensor([[1, 4],
        [2, 5],
        [3, 6]])


## 4.flatten

In [30]:
t = torch.rand(2,3,4)
print(t)

tensor([[[0.2210, 0.7565, 0.6009, 0.0696],
         [0.5066, 0.9365, 0.8618, 0.5284],
         [0.2877, 0.4028, 0.7559, 0.2622]],

        [[0.4206, 0.9912, 0.1087, 0.2305],
         [0.2572, 0.3832, 0.9689, 0.0499],
         [0.1063, 0.4116, 0.6465, 0.9453]]])


In [31]:
print('全部展平',t.flatten())
print('从第二维度开始展平',t.flatten(1))

全部展平 tensor([0.2210, 0.7565, 0.6009, 0.0696, 0.5066, 0.9365, 0.8618, 0.5284, 0.2877,
        0.4028, 0.7559, 0.2622, 0.4206, 0.9912, 0.1087, 0.2305, 0.2572, 0.3832,
        0.9689, 0.0499, 0.1063, 0.4116, 0.6465, 0.9453])
从第二维度开始展平 tensor([[0.2210, 0.7565, 0.6009, 0.0696, 0.5066, 0.9365, 0.8618, 0.5284, 0.2877,
         0.4028, 0.7559, 0.2622],
        [0.4206, 0.9912, 0.1087, 0.2305, 0.2572, 0.3832, 0.9689, 0.0499, 0.1063,
         0.4116, 0.6465, 0.9453]])


## 5.cat和stack

In [32]:
a = torch.ones(2, 3)
b = torch.zeros(2, 3)
print(a)
print(b)

tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0., 0., 0.],
        [0., 0., 0.]])


In [34]:
# cat沿已有维度拼接不增加维度
print(torch.cat([a,b],dim=0))
print(torch.cat([a,b],dim=1))

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.]])


In [35]:
# stack创建新维度再拼接
print(torch.stack([a,b],dim=0))

tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[0., 0., 0.],
         [0., 0., 0.]]])


In [36]:
print(torch.stack([a,b],dim = 1))

tensor([[[1., 1., 1.],
         [0., 0., 0.]],

        [[1., 1., 1.],
         [0., 0., 0.]]])


## 6.拆分split和chunk

In [37]:
t = torch.arange(12).reshape(3,4)
print(t)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


In [38]:
parts = torch.split(t,2,dim=1)
print([part for part in parts])

[tensor([[0, 1],
        [4, 5],
        [8, 9]]), tensor([[ 2,  3],
        [ 6,  7],
        [10, 11]])]


In [39]:
parts = torch.split(t,2,dim=0)
print([part for part in parts])

[tensor([[0, 1, 2, 3],
        [4, 5, 6, 7]]), tensor([[ 8,  9, 10, 11]])]


In [40]:
# chunk：尽量均等地拆成 N 份
chunks = torch.chunk(t, 2, dim=1)       # 沿行拆成 2 份
for c in chunks:
    print(c.shape)

torch.Size([3, 2])
torch.Size([3, 2])


## 7.expend 和 repeat


In [42]:
x = torch.tensor([[1],[2],[3]])
# expand：不复制数据，沿大小为 1 的维度广播
print(x.expand(3, 4))            # (3, 4)

# repeat：真实复制数据
print(x.repeat(1, 4))            # (3, 4)
print(x.repeat(2, 3))            # (6, 3)

tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])
tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])
tensor([[1, 1, 1],
        [2, 2, 2],
        [3, 3, 3],
        [1, 1, 1],
        [2, 2, 2],
        [3, 3, 3]])
